In [15]:
import numpy as np
from numba import njit, prange, float32, uint8
from src.constants import *
import time


In [16]:
N = len(NEURON_NAMES)
ALPHA_L = 250


# Precalculating full arrays for inputs and alphas

# alpha kernel ---------------------------------------------------------         # Use 0 to start at 0.0, 1 to start at .087
td = np.arange(1, ALPHA_L + 1, dtype=np.float32) # Use 0 to start at 0.0, 1 to start at .087
alpha = (td / 30) * np.exp((30 - td) / 30)   # not normalised

cue_wave = np.zeros(TMAX, dtype=np.float32)
go_wave = np.zeros_like(cue_wave)
cue_wave[EPOCHS['sample'][0]:EPOCHS['sample'][1]] = CUE_STRENGTH
go_wave[EPOCHS['response'][0]:EPOCHS['response'][0] + GO_DURATION] = GO_STRENGTH


# --------------------------------------------------------------------
# Hand‑crafted weights -------------------------------------------------
# --------------------------------------------------------------------
new_jh_weights = [
    ("Somat", "ALMprep", 40),
    ("Somat", "MSN1", 220),
    ("MSN1", "SNR1", -90),
    ("SNR1", "VMprep", -10),
    ("VMprep", "ALMprep", 70),
    ("ALMprep", "VMprep", 80),
    ("ALMprep", "MSN2", 320),
    ("MSN2", "SNR2", -50),
    ("SNR2", "VMresp", -100),
    ("PPN", "THALgo", 60),
    ("THALgo", "ALMinter", 55),
    ("ALMinter", "ALMprep", -50),
    ("THALgo", "ALMresp", 30),
    ("ALMresp", "MSN3", 320),
    ("MSN3", "SNR3", -90),
    ("SNR3", "VMresp", -50),
    ("VMresp", "ALMresp", 85),
    ("ALMresp", "VMresp", 90),
]


In [17]:
# CREATING CRITERION

# criterion template for ONE neuron (example) -------------------------
desired_row = np.zeros(NBINS, np.uint8)
desired_row[2:6] = 1               # should spike bins 2-5

NameError: name 'NBINS' is not defined

In [18]:
# Creating Weight Matrix (JH WEIGHTS)

# --------------------------------------------------------------------
# Build weight matrix -------------------------------------------------
# --------------------------------------------------------------------
N = len(NEURON_NAMES)
W = np.zeros((N, N), dtype=np.float32)
for pre, post, w in new_jh_weights:
    i = NEURON_NAMES.index(pre)
    j = NEURON_NAMES.index(post)
    W[i, j] += w


In [19]:
@njit(parallel=True, fastmath=True, cache=True)
def step_kernel(V, U, Ibuf, t_ptr,
                a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                W, alpha):
    n, L = V.size, alpha.size
    spk  = np.zeros(n, dtype=uint8)

    # integrate -------------------------------------------------------
    for i in prange(n):
        I   = Ibuf[i, t_ptr]
        dV  = (k[i]*(V[i]-vr[i])*(V[i]-vt[i]) - U[i] + I + E[i]) / C[i]
        dU  = a[i]*(b[i]*(V[i]-vr[i]) - U[i])
        V[i] += dV
        U[i] += dU
        if V[i] >= vpeak[i]:
            V[i]  = vreset[i]
            U[i] += d[i]
            spk[i] = 1          # Double check the formula to make sure it aint wonky

    # distribute PSC --------------------------------------------------
    if spk.any():
        post_I = spk.astype(float32) @ W                   # dense GEMV
        t_next = (t_ptr + 1) % L
        for k_shift in range(L):
            Ibuf[:, (t_next + k_shift) % L] += post_I * alpha[k_shift]

    Ibuf[:, t_ptr] = 0.0
    return spk, (t_ptr + 1) % L

In [20]:
# ────────────────────────────────────────────────────────────────────
# 2.  Simulation + scoring
# ────────────────────────────────────────────────────────────────────
def simulate(W, cue_wave, go_wave):
    V = np.full(N, -60.0, np.float32)
    U = np.zeros_like(V)
    Ibuf = np.zeros((N, ALPHA_L), np.float32)
    HIST = np.zeros((N, (TMAX + 7) // 8), np.uint8)

    score = 0
    bit_idx = 0
    t_ptr   = 0
    for t in range(TMAX):
        # inject cue/go as plain current (example: to neuron 0)
        Ibuf[0, t_ptr] += cue_wave[t]
        Ibuf[9, t_ptr] += go_wave[t]

        spk, t_ptr = step_kernel(V, U, Ibuf, t_ptr,
                                 a,b,k,vr,vt,vpeak,vreset,d,C,
                                 W, alpha)

        print(t_ptr, V)

        # bit-pack history
        HIST[:, bit_idx] |= spk << (t % 8)
        if t % 8 == 7:
            bit_idx += 1

    #     # simple epoch scoring for neuron 0 ---------------------------
    #     desired = desired_row[t // BIN_MS]
    #     score  += int((spk[0] > 0) == desired)

    # return score, HIST



In [21]:

# ────────────────────────────────────────────────────────────────────
# 3.  Example run
# ────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    t0 = time.time()
    simulate(W, cue_wave, go_wave)
    # score, hist = simulate(W, cue_wave, go_wave)
    # print(f"TOTAL score: {score}")
    print(f"Wall-time: {1000*(time.time()-t0):.1f} ms")

    # unpack if needed
    # bits = np.unpackbits(hist, axis=1)[:, :TMAX]   # shape (N,TMAX)
    # print("Neuron-0 first 40 ms spikes:", bits[0, :40].tolist())

NameError: name 'a' is not defined

In [ ]:
# Testing step kernel


In [ ]:

def simulate(W, cue_wave, go_wave, alpha, T=5000):
    V,U = V0.copy(), U0.copy()          # const templates
    Ibuf= np.zeros((N, alpha.size), np.float32)
    t_ptr = 0
    exp_score = ctrl_score = 0

    for ctl in (False, True):
        # inject exogenous currents
        if ctl:
            Ibuf[ppn_idx, cue_on:cue_off] += go_wave  # example
        else:
            Ibuf[somat_idx, sample_on:sample_off] += cue_wave

        for _ in range(T):
            spk, t_ptr = step_kernel(
                V,U,Ibuf,W,alpha,t_ptr,a,b,k,vr,vt,vpeak,vreset,d,C
            )
            # epoch-based scoring here with simple counters → O(1)

        if ctl:
            ctrl_score = current_score
        else:
            exp_score  = current_score
    return exp_score + ctrl_score//2


In [45]:
# Creating parameter tuples for each neuron
a,b,k,vr,vt,vpeak,vreset,d,C = (
    np.where(TYPE_IDX, PARAM_MSN[i], PARAM_RS[i]).astype(np.float32)
    for i in range(9) ## DOUBLE CHECK TO MAKE SURE EVERY CONSTANT IS HERE
)

print(a,b,k,vr,vt,vpeak,vreset,d,C, sep="\n")

[0.03 0.01 0.03 0.03 0.03 0.01 0.03 0.03 0.03 0.03 0.01 0.03 0.03 0.03]
[ -2. -20.  -2.  -2.  -2. -20.  -2.  -2.  -2.  -2. -20.  -2.  -2.  -2.]
[-50. -55. -50. -50. -50. -55. -50. -50. -50. -50. -55. -50. -50. -50.]
[100. 150. 100. 100. 100. 150. 100. 100. 100. 100. 150. 100. 100. 100.]
[0.7 1.  0.7 0.7 0.7 1.  0.7 0.7 0.7 0.7 1.  0.7 0.7 0.7]
[-60. -80. -60. -60. -60. -80. -60. -60. -60. -60. -80. -60. -60. -60.]
[-40. -25. -40. -40. -40. -25. -40. -40. -40. -40. -25. -40. -40. -40.]
[35. 40. 35. 35. 35. 40. 35. 35. 35. 35. 40. 35. 35. 35.]
[ 0. 70.  0.  0.  0. 70.  0.  0.  0.  0. 70.  0.  0.  0.]
